In [ ]:
%py
# Mask the last four digits of invoice_number in purgo_playground.d_product_revenue_clone

from pyspark.sql.functions import col, when, concat, lit, substring
from pyspark.sql.types import StringType

# Commented out SparkSession initialization as spark is already available in Databricks
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("InvoiceNumberMasking").getOrCreate()

try:
    # Drop the clone table if it exists
    spark.sql("""
        DROP TABLE IF EXISTS purgo_playground.d_product_revenue_clone
    """)
except Exception as e:
    # Log the exception
    print(f"Error dropping table purgo_playground.d_product_revenue_clone: {e}")

try:
    # Create the clone table from the source table
    spark.sql("""
        CREATE TABLE purgo_playground.d_product_revenue_clone AS
        SELECT * FROM purgo_playground.d_product_revenue
    """)
except Exception as e:
    # Log the exception
    print(f"Error creating table purgo_playground.d_product_revenue_clone: {e}")

try:
    # Read the clone table
    df = spark.table("purgo_playground.d_product_revenue_clone")
    
    # Convert invoice_number to string and handle nulls
    df_masked = df.withColumn(
        "invoice_number",
        when(
            col("invoice_number").isNotNull(),
            when(
                length(col("invoice_number").cast(StringType())) > 4,
                concat(
                    substring(col("invoice_number").cast(StringType()), 1, length(col("invoice_number").cast(StringType())) - 4),
                    lit("****")
                )
            ).otherwise(
                concat(lit("*"), lit("*"), lit("*"), lit("*")).substr(1, length(col("invoice_number").cast(StringType())))
            )
        ).otherwise(col("invoice_number"))
    )
    
    # Overwrite the clone table with masked invoice_number
    df_masked.write.mode("overwrite").saveAsTable("purgo_playground.d_product_revenue_clone")
except Exception as e:
    # Log the exception
    print(f"Error masking invoice_number in purgo_playground.d_product_revenue_clone: {e}")

# Commented out spark.stop() to prevent issues in Databricks
# spark.stop()